# Triple Extraction with LLMs: AI-Powered Knowledge Graph Construction

LLMs offer a fundamentally different approach to triple extraction compared to NLP pipelines (see Notebook 03):

| Aspect | spaCy / NLP | LLM-based |
|--------|------------|----------|
| Setup | Needs labeled training data | Zero/few-shot with prompts |
| Recall | Lower — misses implicit relations | Higher — captures paraphrased and long-range relations |
| Precision | High for known patterns | Risk of hallucination |
| Cost | Fast, cheap | Slower, more expensive per token |

This notebook demonstrates three LLM extraction patterns:
1. Single-pass instruction prompting
2. Two-step extraction (entities first, then relations)
3. Schema-constrained extraction

> **Note**: This notebook uses the OpenAI API. If no API key is available, simulated outputs are provided so you can still follow along.

In [ ]:
%pip install openai networkx matplotlib pandas -q

In [ ]:
import os
import json
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

# Check for OpenAI API key
USE_LIVE_API = bool(os.environ.get("OPENAI_API_KEY"))

if USE_LIVE_API:
    from openai import OpenAI
    client = OpenAI()
    print("OpenAI API key found. Using live API calls.")
else:
    print("No OPENAI_API_KEY found. Using simulated outputs.")
    print("Set your API key: export OPENAI_API_KEY='your-key-here'")

In [ ]:
# Sample text for all extraction methods
SAMPLE_TEXT = """Geoffrey Hinton joined Google Brain in 2013. He is known as the godfather 
of deep learning. Hinton received the Turing Award in 2018 along with Yann LeCun 
and Yoshua Bengio. LeCun leads AI research at Meta, while Bengio co-founded 
Mila institute in Montreal. DeepMind, founded by Demis Hassabis in London, 
was acquired by Google in 2014."""

print("Source text:")
print(SAMPLE_TEXT)

## 1. Single-Pass Instruction Prompting

The simplest approach: send the text to an LLM with instructions to extract triples in a structured format.

In [ ]:
SINGLE_PASS_PROMPT = """Extract all knowledge graph triples from the following text. 
Return a JSON array where each element has 'subject', 'predicate', and 'object' fields.
Only extract facts explicitly stated in the text. Be precise.

Text: {text}

Return ONLY valid JSON, no other text."""

# Simulated output (used when API key is not available)
SIMULATED_SINGLE_PASS = [
    {"subject": "Geoffrey Hinton", "predicate": "joined", "object": "Google Brain"},
    {"subject": "Geoffrey Hinton", "predicate": "joined_in", "object": "2013"},
    {"subject": "Geoffrey Hinton", "predicate": "known_as", "object": "godfather of deep learning"},
    {"subject": "Geoffrey Hinton", "predicate": "received", "object": "Turing Award"},
    {"subject": "Yann LeCun", "predicate": "received", "object": "Turing Award"},
    {"subject": "Yoshua Bengio", "predicate": "received", "object": "Turing Award"},
    {"subject": "Turing Award", "predicate": "awarded_in", "object": "2018"},
    {"subject": "Yann LeCun", "predicate": "leads", "object": "AI research at Meta"},
    {"subject": "Yoshua Bengio", "predicate": "co-founded", "object": "Mila"},
    {"subject": "Mila", "predicate": "located_in", "object": "Montreal"},
    {"subject": "DeepMind", "predicate": "founded_by", "object": "Demis Hassabis"},
    {"subject": "DeepMind", "predicate": "founded_in", "object": "London"},
    {"subject": "DeepMind", "predicate": "acquired_by", "object": "Google"},
    {"subject": "DeepMind", "predicate": "acquired_in", "object": "2014"}
]

def extract_single_pass(text):
    if USE_LIVE_API:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": SINGLE_PASS_PROMPT.format(text=text)}],
            temperature=0
        )
        return json.loads(response.choices[0].message.content)
    else:
        return SIMULATED_SINGLE_PASS

single_pass_triples = extract_single_pass(SAMPLE_TEXT)

print(f"Extracted {len(single_pass_triples)} triples:\n")
df_sp = pd.DataFrame(single_pass_triples)
df_sp

## 2. Two-Step Extraction (Entities First, Then Relations)

A more structured approach inspired by systems like **KGGen**:
1. **Step 1**: Extract and type all entities
2. **Step 2**: Given the entities, extract relations between them

This reduces hallucinated entities because the LLM is constrained to use only entities from Step 1.

In [ ]:
ENTITY_PROMPT = """Extract all named entities from this text. 
Return a JSON array where each element has 'name' and 'type' fields.
Types should be: Person, Organization, Award, Location, Date, Field.

Text: {text}

Return ONLY valid JSON."""

RELATION_PROMPT = """Given these entities and the source text, extract relationships between them.
Return a JSON array with 'subject', 'predicate', 'object' fields.
ONLY use entity names from the provided list. Do not invent new entities.

Entities: {entities}

Text: {text}

Return ONLY valid JSON."""

SIMULATED_ENTITIES = [
    {"name": "Geoffrey Hinton", "type": "Person"},
    {"name": "Google Brain", "type": "Organization"},
    {"name": "Yann LeCun", "type": "Person"},
    {"name": "Yoshua Bengio", "type": "Person"},
    {"name": "Demis Hassabis", "type": "Person"},
    {"name": "Meta", "type": "Organization"},
    {"name": "Mila", "type": "Organization"},
    {"name": "DeepMind", "type": "Organization"},
    {"name": "Google", "type": "Organization"},
    {"name": "Turing Award", "type": "Award"},
    {"name": "Montreal", "type": "Location"},
    {"name": "London", "type": "Location"},
    {"name": "deep learning", "type": "Field"}
]

SIMULATED_RELATIONS = [
    {"subject": "Geoffrey Hinton", "predicate": "joined", "object": "Google Brain"},
    {"subject": "Geoffrey Hinton", "predicate": "known_as_godfather_of", "object": "deep learning"},
    {"subject": "Geoffrey Hinton", "predicate": "received", "object": "Turing Award"},
    {"subject": "Yann LeCun", "predicate": "received", "object": "Turing Award"},
    {"subject": "Yoshua Bengio", "predicate": "received", "object": "Turing Award"},
    {"subject": "Yann LeCun", "predicate": "leads_AI_research_at", "object": "Meta"},
    {"subject": "Yoshua Bengio", "predicate": "co-founded", "object": "Mila"},
    {"subject": "Mila", "predicate": "located_in", "object": "Montreal"},
    {"subject": "Demis Hassabis", "predicate": "founded", "object": "DeepMind"},
    {"subject": "DeepMind", "predicate": "located_in", "object": "London"},
    {"subject": "Google", "predicate": "acquired", "object": "DeepMind"}
]

def extract_two_step(text):
    if USE_LIVE_API:
        # Step 1: Extract entities
        ent_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": ENTITY_PROMPT.format(text=text)}],
            temperature=0
        )
        entities = json.loads(ent_response.choices[0].message.content)
        
        # Step 2: Extract relations
        entity_list = json.dumps([e["name"] for e in entities])
        rel_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": RELATION_PROMPT.format(
                entities=entity_list, text=text)}],
            temperature=0
        )
        relations = json.loads(rel_response.choices[0].message.content)
        return entities, relations
    else:
        return SIMULATED_ENTITIES, SIMULATED_RELATIONS

entities, two_step_triples = extract_two_step(SAMPLE_TEXT)

print("Step 1 — Entities:")
pd.DataFrame(entities)

In [ ]:
print(f"Step 2 — Relations ({len(two_step_triples)} triples):\n")
df_ts = pd.DataFrame(two_step_triples)
df_ts

## 3. Schema-Constrained Extraction

By providing a **schema** (allowed entity types and relation types), we constrain the LLM to produce consistent, ontology-aligned triples.

In [ ]:
schema = {
    "entity_types": ["Person", "Organization", "Award", "Location", "Field"],
    "relation_types": [
        "works_at", "founded", "co_founded", "received_award", 
        "located_in", "acquired", "researches", "leads"
    ]
}

SCHEMA_PROMPT = """Extract knowledge graph triples from the text below.
You MUST use only these entity types: {entity_types}
You MUST use only these relation types: {relation_types}

Return a JSON array where each element has:
- 'subject': entity name
- 'subject_type': one of the allowed entity types
- 'predicate': one of the allowed relation types  
- 'object': entity name
- 'object_type': one of the allowed entity types

Text: {text}

Return ONLY valid JSON."""

SIMULATED_SCHEMA = [
    {"subject": "Geoffrey Hinton", "subject_type": "Person", "predicate": "works_at", "object": "Google Brain", "object_type": "Organization"},
    {"subject": "Geoffrey Hinton", "subject_type": "Person", "predicate": "received_award", "object": "Turing Award", "object_type": "Award"},
    {"subject": "Yann LeCun", "subject_type": "Person", "predicate": "received_award", "object": "Turing Award", "object_type": "Award"},
    {"subject": "Yoshua Bengio", "subject_type": "Person", "predicate": "received_award", "object": "Turing Award", "object_type": "Award"},
    {"subject": "Yann LeCun", "subject_type": "Person", "predicate": "leads", "object": "Meta", "object_type": "Organization"},
    {"subject": "Yoshua Bengio", "subject_type": "Person", "predicate": "co_founded", "object": "Mila", "object_type": "Organization"},
    {"subject": "Mila", "subject_type": "Organization", "predicate": "located_in", "object": "Montreal", "object_type": "Location"},
    {"subject": "Demis Hassabis", "subject_type": "Person", "predicate": "founded", "object": "DeepMind", "object_type": "Organization"},
    {"subject": "DeepMind", "subject_type": "Organization", "predicate": "located_in", "object": "London", "object_type": "Location"},
    {"subject": "Google", "subject_type": "Organization", "predicate": "acquired", "object": "DeepMind", "object_type": "Organization"},
    {"subject": "Geoffrey Hinton", "subject_type": "Person", "predicate": "researches", "object": "deep learning", "object_type": "Field"}
]

def extract_schema_constrained(text, schema):
    if USE_LIVE_API:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": SCHEMA_PROMPT.format(
                entity_types=schema["entity_types"],
                relation_types=schema["relation_types"],
                text=text
            )}],
            temperature=0
        )
        return json.loads(response.choices[0].message.content)
    else:
        return SIMULATED_SCHEMA

schema_triples = extract_schema_constrained(SAMPLE_TEXT, schema)

print(f"Schema-constrained extraction: {len(schema_triples)} triples\n")
df_schema = pd.DataFrame(schema_triples)
df_schema

## 4. Visualizing the Knowledge Graph

Let's build a graph from the schema-constrained triples (the most structured output).

In [ ]:
G = nx.DiGraph()
node_types = {}

for t in schema_triples:
    G.add_edge(t["subject"], t["object"], relation=t["predicate"])
    node_types[t["subject"]] = t.get("subject_type", "Unknown")
    node_types[t["object"]] = t.get("object_type", "Unknown")

type_colors = {
    "Person": "#FF9999", "Organization": "#99CCFF", 
    "Award": "#CC99FF", "Location": "#99FF99", 
    "Field": "#FFCC99"
}
color_map = [type_colors.get(node_types.get(n, ""), "#D9D9D9") for n in G.nodes()]

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42, k=2)
nx.draw(G, pos, with_labels=True, node_color=color_map,
        node_size=2800, font_size=9, font_weight="bold",
        arrows=True, arrowsize=15, edge_color="gray")

edge_labels = nx.get_edge_attributes(G, "relation")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

legend_handles = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()]
plt.legend(handles=legend_handles, loc="upper left", fontsize=10)
plt.title("Knowledge Graph from LLM Extraction (Schema-Constrained)", fontsize=14)
plt.tight_layout()
plt.show()

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Density: {nx.density(G):.3f}")

## 5. Handling Hallucinations: Validation

A key weakness of LLM extraction is **hallucination** — the model may invent entities or relations not present in the source text. A simple validation step checks:
1. Do subject and object appear in (or closely match) the source text?
2. Is the predicate from the allowed schema?

In [ ]:
def validate_triples(triples, source_text, allowed_predicates=None):
    """Validate extracted triples against source text."""
    source_lower = source_text.lower()
    results = []
    
    for t in triples:
        subj_in_text = t["subject"].lower() in source_lower
        obj_in_text = t["object"].lower() in source_lower
        pred_valid = True
        if allowed_predicates:
            pred_valid = t["predicate"] in allowed_predicates
        
        is_valid = subj_in_text and obj_in_text and pred_valid
        results.append({
            "subject": t["subject"],
            "predicate": t["predicate"],
            "object": t["object"],
            "subj_found": subj_in_text,
            "obj_found": obj_in_text,
            "pred_valid": pred_valid,
            "VALID": is_valid
        })
    return results

# Validate schema-constrained triples
validation = validate_triples(schema_triples, SAMPLE_TEXT, schema["relation_types"])
df_val = pd.DataFrame(validation)

valid_count = df_val["VALID"].sum()
total = len(df_val)
print(f"Validation: {valid_count}/{total} triples passed ({valid_count/total:.0%})\n")
df_val

In [ ]:
# Show only failed triples (if any)
failed = df_val[~df_val["VALID"]]
if len(failed) > 0:
    print("Failed validation:")
    print(failed[["subject", "predicate", "object", "subj_found", "obj_found", "pred_valid"]])
else:
    print("All triples passed validation!")

## 6. Comparing Extraction Approaches

| Approach | Triples | Pros | Cons |
|----------|---------|------|------|
| **Single-pass** | Highest count | Simple, fast, good recall | May hallucinate, inconsistent predicates |
| **Two-step** | Moderate | Entities constrained, cleaner | Two API calls, slightly slower |
| **Schema-constrained** | Controlled | Consistent types & predicates, ontology-aligned | May miss relations outside schema |

### Recommendations for Agentic Systems

- **Use schema-constrained extraction** when you have a defined ontology (most common in production)
- **Add validation** to catch hallucinations — check that entities appear in source text
- **Hybrid approach**: Use spaCy for fast entity detection, LLM for flexible relation extraction
- **Consider cost**: For large corpora, batch processing and careful prompt design reduce API costs

### The Hybrid Pipeline (Best Practice)

```
Text → spaCy (NER) → Entity List → LLM (Relations only) → Validation → Knowledge Graph
```

This combines spaCy's speed and reliability for entity detection with the LLM's semantic flexibility for relation extraction, while validation catches any remaining errors.